# Create IMLS Awards (Institute of Museum and Library Services)

Creates Institute of Museum and Library Services awards from IMLS's own
"Awarded Grants" search at https://www.imls.gov/grants/awarded-grants, using
the page's **Download the Data** CSV export (Drupal views_data_export; ladder
step 0, the funder's own bulk export). The export covers every IMLS award
FY1997-FY2026: Log Number, Description, Supplement Indicator, Institution,
Fiscal Year, Program, Federal Funds, Funding Office, City, State.
**21,037 awards, FY1997-2026, USD 2.17B, 100% institution/amount/fiscal year,
51% description** (local run 2026-10-01).

**Scope filter (oxjob #1451, 2026-10-01 rule):** excludes ONLY the formula
Grants to States (LSTA population-based allotments to State Library
Administrative Agencies, which are state operating money): programmes
"Grants to State Library Administrative Agencies", "Grants to State ARPA State
Grants", "Grants to State CARES ACT State Grants" (1,883 LS-* log numbers,
filtered in the script, never reach this table). All other programmes are kept,
including the non-competitive Native American Library Services Basic Grants
(5,432) and the 1990s museum General Operating Support (1,107).

**No title, no named PI.** Neither the export nor the grant pages publish a
project title or project director. `display_name` is composed from the source's
own fields: `{Program} — {Institution} (FY{year})`. Awards are org-level:
`lead_investigator` carries the recipient institution only (DOE Office of
Science precedent).

**Dates:** IMLS publishes only the award fiscal year. `start_year` = fiscal year
of the base award; `start_date` = Jan 1 of that year (runbook §2.3 year-only
pattern; IMLS awards are made in the summer of the fiscal year, so the calendar
year matches). No end date is published.

**Supplements:** the export lists each supplement as an extra row under the
original log number; the script folds them into one award per log number
(amount = base + supplements; `amount_base` / `amount_supplements` kept in raw).

**`funder_award_id` (§2.1.1):** the IMLS log number (e.g. `LG-252287-OLS-22`,
`RE-04-12-0105-12`), the reference grantees cite. Citation-side sample
(`openalex_awards_raw` stubs for F4320306122): 465 of 833 stubs match a log number
exactly (case-insensitive), so those collapse onto these awards; most of the rest
are typographic variants (unicode dashes, dropped `-YY` suffix, prefixes like
"IMLS ").

**Overlap check:** no USAspending ingest covers IMLS (grep of notebooks/awards and
scripts/local), and no existing direct ingest uses F4320306122. No twin funder row
(same name / ROR 030prv062).

**Prerequisites:** run `scripts/local/imls_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/imls/imls_projects.parquet` (§1.4 shrink guard).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320306122`
- display_name: Institute of Museum and Library Services
- ror_id / doi: from `openalex.funders.funders` (ror 030prv062, doi 10.13039/100000208)

**Priority:** `536` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.imls_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/imls/imls_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT log_number) as distinct_log_numbers FROM openalex.awards.imls_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.imls_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.imls_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320306122 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320306122;

## Step 2: Create IMLS Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.imls_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320306122  -- Institute of Museum and Library Services
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.log_number))))) % 9000000000 as id,
    -- no title in the source: compose from the source's own programme / recipient / year
    CONCAT(TRIM(g.program), ' — ', TRIM(g.institution), ' (FY', g.fiscal_year, ')') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.log_number) as funder_award_id,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) END as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type as funding_type,
    NULLIF(TRIM(g.program), '') as funder_scheme,
    'imls_awarded_grants' as provenance,
    TRY_TO_DATE(CONCAT(g.fiscal_year, '-01-01'), 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.fiscal_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- org-level award: recipient institution only (no named project director published).
    -- struct field order MUST match openalex_awards_raw.
    CASE WHEN NULLIF(TRIM(g.institution), '') IS NOT NULL THEN named_struct(
        'given_name', CAST(NULL AS STRING),
        'family_name', CAST(NULL AS STRING),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', TRIM(g.institution),
            -- Freely Associated States are IMLS-eligible but not US
            'country', CASE WHEN g.state IN ('FM', 'MH', 'PW') THEN g.state
                            WHEN g.state IS NOT NULL THEN 'US' END,
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.log_number))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.imls_raw g
CROSS JOIN src_funder f
WHERE g.log_number IS NOT NULL AND TRIM(g.log_number) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): every funder_award_id is an IMLS log number; expect 0 rows.
SELECT funder_award_id FROM openalex.awards.imls_awards
WHERE NOT funder_award_id RLIKE '^[0-9A-Z]{2,6}-[0-9]{2,6}-'
LIMIT 20;

In [ ]:
%sql
-- Never shrink what may already be published: refuse to replace a larger earlier load.
SELECT assert_true(
    (SELECT COUNT(*) FROM openalex.awards.imls_awards) >=
    (SELECT COUNT(*) FROM openalex.awards.openalex_awards_raw WHERE provenance = 'imls_awarded_grants' AND priority = 536),
    'imls_awards is smaller than the rows already in openalex_awards_raw; refusing to shrink'
) as no_shrink;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'imls_awarded_grants' AND priority = 536;

-- Insert into openalex_awards_raw with priority.
-- Priority 536: direct-from-funder ingest of IMLS's own awarded-grants export.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 536 outranks the
-- crossref/europepmc/datacite acknowledgement shells (priority 0) sharing these ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    536 as priority
FROM openalex.awards.imls_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_imls_awards, COUNT(DISTINCT id) as distinct_ids FROM openalex.awards.imls_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.affiliation.name, lead_investigator.affiliation.country, landing_page_url
FROM openalex.awards.imls_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.imls_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.imls_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check (org-level: recipient institution and display_name).
SELECT lead_investigator.affiliation.name AS institution, COUNT(*) AS n
FROM openalex.awards.imls_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.imls_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (expect 100% title/amount/start_year/institution, ~51% description).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.imls_awards;

In [ ]:
%sql
-- Citation stubs this ingest collapses onto (same award id).
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(t.id) as matched_by_imls_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.imls_awards t ON t.id = c.id
WHERE c.funder_id = 4320306122 AND c.priority < 3
GROUP BY c.provenance ORDER BY stub_rows DESC;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'imls_awarded_grants'
GROUP BY provenance, priority;